# Level 5: Quantization & Efficient Inference

Large Language Models are... large. A 7-billion parameter model in its standard 16-bit precision (float16) requires about 14GB of GPU VRAM just to be loaded, without even considering the memory needed for inference. This puts running large models out of reach for most consumer hardware.

**Quantization** is the solution. It's the process of reducing the numerical precision of a model's weights. By representing the weights with fewer bits (e.g., 8-bit or 4-bit integers instead of 16-bit floats), we can dramatically shrink the model's memory footprint.

### The Benefits of Quantization

- **Reduced Memory Usage**: This is the biggest win. A 4-bit quantized model can be up to 4x smaller than its 16-bit counterpart.
- **Faster Inference**: Calculations with lower-precision numbers can be faster on modern hardware.
- **Accessibility**: Enables running powerful models on consumer-grade GPUs.

The `bitsandbytes` library, integrated with Hugging Face `transformers`, makes this incredibly easy.

### Step 1: Install Dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes torch

### Step 2: Loading a Model in 8-bit Precision

Loading a model in 8-bit is as simple as adding the `load_in_8bit=True` flag. Let's see the memory difference.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model without quantization (requires a lot of VRAM!)
# model_fp16 = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
# print(f"Memory footprint (FP16): {model_fp16.get_memory_footprint() / 1e9:.2f} GB")
# del model_fp16 # Free up memory

# Load the model with 8-bit quantization
model_8bit = AutoModelForCausalLM.from_pretrained(model_id, load_in_8bit=True, device_map="auto")
print(f"Memory footprint (8-bit): {model_8bit.get_memory_footprint() / 1e9:.2f} GB")

# Test inference
prompt = "You are a helpful assistant. What is quantization?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model_8bit.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Step 3: Loading a Model in 4-bit Precision

4-bit quantization offers even greater memory savings. It requires a `BitsAndBytesConfig` object where we specify the details of the quantization.

Key parameters:
- `load_in_4bit=True`: The magic flag.
- `bnb_4bit_quant_type="nf4"`: Specifies the quantization data type. 'nf4' (Normal Float 4) is a special data type designed for normally distributed weights and is highly effective.
- `bnb_4bit_compute_dtype=torch.float16`: While the weights are stored in 4-bit, computations are performed in a higher precision (like 16-bit float) for better accuracy.

In [ ]:
from transformers import BitsAndBytesConfig

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the model with 4-bit quantization
model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Memory footprint (4-bit): {model_4bit.get_memory_footprint() / 1e9:.2f} GB")

# Test inference
prompt = "You are a helpful assistant. What is 4-bit quantization?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model_4bit.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Trade-offs

Quantization is not free. There is a small, often negligible, cost in terms of model performance (e.g., a slight increase in perplexity or a decrease in benchmark scores). However, for most practical applications, the massive savings in memory and potential for faster inference far outweigh the minimal loss in accuracy.